# 02 · Chain-of-Thought Prompting

你有没有发现，LLM 做数学题或逻辑推理时经常出错？
但如果你让它「先想一想再回答」，准确率会大幅提升。

这就是 **Chain-of-Thought（CoT，思维链）**：
让模型在给出最终答案之前，先输出推理步骤。

本章从原理出发，用实验验证 CoT 的效果，并介绍几种变体。

In [ ]:
from utils.llm_client import chat

## 1. 为什么需要 CoT

LLM 生成 token 的方式是**自回归**的——每个 token 只依赖之前的 token。
对于需要多步推理的问题，如果直接跳到答案，中间推理步骤没有被「写出来」，
模型就无法在它们之上继续推理。

CoT 的本质：**把「思考过程」变成 token，让模型能在推理步骤上继续推理。**

In [ ]:
# 经典推理题：不用 CoT vs 用 CoT
problem = """
小明有 5 个苹果，他给了小红 2 个，然后又从商店买了 3 个，
回家路上掉了 1 个，最后到家时还剩多少个苹果？
""".strip()

# 直接回答
direct = chat(f"{problem}\n\n直接给出数字答案。", temperature=0)
print("直接回答:", direct.strip())

# CoT
cot = chat(f"{problem}\n\n请一步一步推理，然后给出答案。", temperature=0)
print("\nCoT 回答:")
print(cot.strip())

## 2. Zero-shot CoT

最简单的 CoT：在 prompt 末尾加上魔法短语——

> **"Let's think step by step."**

这是 Kojima et al. 2022 的发现：仅这一句话就能显著提升推理准确率，无需任何示例。

In [ ]:
def solve(problem: str, cot: bool = False) -> str:
    suffix = "\n\nLet's think step by step." if cot else "\n\n直接给出答案，不要解释。"
    return chat(problem + suffix, temperature=0).strip()

problems = [
    {
        "题目": "逻辑题",
        "问题": "所有玫瑰都是花。有些花会凋谢。所以，有些玫瑰会凋谢。这个推理正确吗？",
        "正确答案": "不一定正确",
    },
    {
        "题目": "数学题",
        "问题": "一个水池，单独用A管需要6小时注满，单独用B管需要4小时注满。两管同时开，需要多少小时？",
        "正确答案": "2.4小时",
    },
    {
        "题目": "文字题",
        "问题": """如果 'ABCD' 反过来写是 'DCBA'，那么 'LOGIC' 反过来写，第3个字母是什么？""",
        "正确答案": "I",
    },
]

for p in problems:
    ans_direct = solve(p["问题"], cot=False)
    ans_cot    = solve(p["问题"], cot=True)
    print(f"\n{'='*55}")
    print(f"题目: {p['题目']}")
    print(f"问题: {p['问题']}")
    print(f"正确答案:  {p['正确答案']}")
    print(f"直接回答:  {ans_direct[:100]}")
    print(f"CoT 回答:\n{ans_cot[:300]}")

## 3. Few-shot CoT

在 few-shot 的例子里，连带推理步骤一起给出。
模型会学习「先推理，再给答案」的模式，效果比 zero-shot CoT 更稳定。

In [ ]:
few_shot_cot_prompt = """
解答数学应用题，先写推理过程，最后一行写「答案：X」。

问题：小红有12颗糖，吃了3颗，又买了6颗，最后有多少颗？
推理：
  初始：12颗
  吃了3颗：12 - 3 = 9颗
  又买了6颗：9 + 6 = 15颗
答案：15

问题：一辆车时速60公里，行驶了2.5小时，中途休息了30分钟，实际行驶了多少公里？
推理：
  总时间：2.5小时
  休息时间：30分钟 = 0.5小时
  实际行驶时间：2.5 - 0.5 = 2小时
  行驶距离：60 × 2 = 120公里
答案：120

问题：班级有40名学生，60%是女生，女生中有25%参加了合唱团，合唱团有多少女生？
推理：
""".strip()

result = chat(few_shot_cot_prompt, temperature=0)
print(result.strip())

## 4. Self-Consistency（自洽性）

单次 CoT 可能因采样随机性而出错。
**Self-consistency** 的做法：
1. 用高 temperature 跑多次 CoT
2. 收集所有答案
3. **多数票** 作为最终答案

Wang et al. 2023 证明这能显著提升准确率，尤其在数学和推理任务上。

In [ ]:
import re
from collections import Counter

def extract_answer(text: str) -> str:
    """从 CoT 输出中提取最终答案（最后一个数字）。"""
    numbers = re.findall(r"\d+\.?\d*", text)
    return numbers[-1] if numbers else text.strip()[-10:]

def self_consistency(problem: str, n: int = 5, temperature: float = 0.7) -> dict:
    """多次采样 CoT，取多数票作为最终答案。"""
    prompt = problem + "\n\nLet's think step by step."
    answers = []
    for i in range(n):
        response = chat(prompt, temperature=temperature)
        ans = extract_answer(response)
        answers.append(ans)
        print(f"  第{i+1}次: {ans}")

    vote = Counter(answers).most_common(1)[0]
    return {"answers": answers, "majority": vote[0], "count": vote[1]}

problem = "一个工程队修一条路，前3天修了全长的1/4，照这样的速度，修完全部还需要多少天？"
print(f"题目: {problem}\n")
print(f"正确答案: 9天\n")
print("5次采样结果:")
result = self_consistency(problem, n=5)
print(f"\n多数票答案: {result['majority']}（{result['count']}/5 票）")

## 5. CoT 的实用技巧

In [ ]:
# 技巧 1：强制在答案前输出推理，用分隔符分开
# 方便程序解析「推理」和「答案」两部分
structured_cot_prompt = """分析以下客户评论，判断情感并给出理由。
格式严格按照：
推理：<分析过程>
答案：<正面/负面/中性>

评论：产品质量不错，但客服态度很差，下次不知道还会不会买。"""

response = chat(structured_cot_prompt, temperature=0)
print(response.strip())

# 解析答案
lines = response.strip().split("\n")
for line in lines:
    if line.startswith("答案："):
        print(f"\n程序解析到的答案: {line.replace('答案：', '').strip()}")

In [ ]:
# 技巧 2：让模型先反驳自己的答案，再给最终结论（更稳健）
debate_prompt = """回答以下问题，要求：
1. 先给出初步答案
2. 列出可能反驳该答案的理由
3. 综合考虑后给出最终答案

问题：「先有鸡还是先有蛋」从进化论角度来看，答案是什么？"""

print(chat(debate_prompt, temperature=0).strip())

## 小结

| 技术 | 核心做法 | 适用场景 |
|------|---------|----------|
| Zero-shot CoT | 加「Let's think step by step」 | 快速提升推理，无需准备例子 |
| Few-shot CoT | 给带推理步骤的示例 | 有特定输出格式要求时 |
| Self-consistency | 多次采样取多数票 | 高准确率要求，可接受多次调用成本 |
| 结构化推理 | 用分隔符分开推理和答案 | 需要程序解析结果时 |

**什么时候不需要 CoT：** 简单的事实查询、格式转换、分类任务——这些用 few-shot 或 zero-shot 直接回答更高效。

**下一章 →** [Structured Output](03_structured_output.ipynb)：用 JSON Mode 和 Function Calling 让模型输出程序可直接解析的结构